# SwinIR Super-Resolution Evaluation

This notebook evaluates one or more **SwinIR super-resolution outputs**
against paired ground-truth (GT) tiles using the repository's existing
evaluation and visualization scripts.

Quantitative evaluation is performed with `evaluate_sr_against_pdok.py`,
which reports:

- **PSNR** — pixel-wise reconstruction fidelity.
- **SSIM** — structural similarity in RGB space.

The notebook is designed for paired SR/GT tile folders and keeps the
workflow close to the scripts already used in the original SwinIR
notebook.

## What this notebook does

- Evaluates one or more SR result folders against a GT tile folder.
- Uses the existing `evaluate_sr_against_pdok.py` utility instead of
  re-implementing the metric logic in the notebook.
- Saves per-image metric values to CSV files.
- Builds a compact summary table with mean / min / max values per model.
- Optionally visualizes a random LR / SR / GT triplet with
  `visualize_random_sr_triplet.py`.

## What this notebook does *not* do

- **Inference** — see `SwinIR_Inference.ipynb`.
- **Training / fine-tuning** — see `SwinIR_Training.ipynb`.
- **Tile stitching / map reconstruction** — run your mapping notebook separately.

## Prerequisites

1. One or more folders with SwinIR SR predictions already exist.
2. A matching GT folder exists with corresponding filenames.
3. If you want qualitative visualization, the LR folder is also available.
4. The repository contains:
   - `evaluate_sr_against_pdok.py`
   - `visualize_random_sr_triplet.py`
5. `opencv-python`, `numpy`, `pandas`, `matplotlib`, and `scikit-image`
   are installed.

## Notes on filename matching

The evaluation and visualization scripts match files by filename stem and
also handle common SR suffixes such as `_SwinIR` and `_x4`. This is useful
because `main_test_swinir.py` saves outputs with names such as
`<image_id>_SwinIR.png`.



In [ ]:
# Imports
from pathlib import Path
import re
import shlex
import subprocess

import numpy as np
import pandas as pd



In [ ]:
# Configuration
# Adjust these paths to your local setup.

REPO_ROOT = Path('/home/datalab/Datalab/swinir_repo') # Change this to your local SwinIR repository path.
EVAL_SCRIPT = REPO_ROOT / 'evaluate_sr_against_pdok.py' # Path to the evaluation script.
VIS_SCRIPT = REPO_ROOT / 'visualize_random_sr_triplet.py' # Path to the visualization script.

# Optional LR directory, only used for the qualitative triplet figure.
LR_DIR = Path('/home/datalab/Datalab/data/inputs') # Change this to your local LR images path.

# Ground-truth HR tiles.
GT_DIR = Path('/home/datalab/Datalab/data/ground_truth') # Change this to your local GT images path.

# One or more SR result folders to compare.
MODEL_DIRS: dict[str, Path] = {
    'SwinIR-L pretrained': REPO_ROOT / 'results' / 'swinir_real_sr_x4_large'
} # Change this to your local SR results path(s).

# Crop border before metric computation. This is common in SR evaluation
# to avoid counting boundary artefacts introduced by upsampling.
CROP_BORDER = 4

# If True, fail when SR and GT images do not have identical dimensions.
# If False, the evaluation script center-crops to a common spatial extent.
STRICT_SIZE = False

# Where to save CSV outputs.
RESULTS_DIR = REPO_ROOT / 'results' / 'evaluation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / 'swinir_evaluation_results.csv'

# Qualitative visualization settings.
VIS_MODEL = 'SwinIR-L pretrained'
VIS_SEED = 42
VIS_SAVE_PATH = RESULTS_DIR / 'swinir_random_triplet.png'

print(f'Repo root : {REPO_ROOT}')
print(f'GT dir    : {GT_DIR}')
print(f'Models    : {list(MODEL_DIRS)}')
print(f'Results   : {RESULTS_DIR}')



## Repository sanity check

Confirm that the required helper scripts exist before running the
evaluation workflow.



In [ ]:
required_files = [EVAL_SCRIPT, VIS_SCRIPT]
missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        'Missing required files:\n' + '\n'.join(str(path) for path in missing_files))

print('Required evaluation files found.')



## Helper functions

The helpers below build and run the repository scripts, collect the
per-image CSV outputs, and combine them into one long-format DataFrame.



In [ ]:
def slugify(name: str) -> str:
    """Convert a model label into a filesystem-friendly slug."""
    text = name.lower().strip()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_')


def build_eval_command(
    script_path: Path,
    folder_sr: Path,
    folder_gt: Path,
    crop_border: int,
    csv_path: Path,
    strict_size: bool = False,
) -> list[str]:
    """Build the evaluation command for one SR folder."""
    cmd = [
        'python',
        str(script_path),
        '--folder_sr', str(folder_sr),
        '--folder_gt', str(folder_gt),
        '--crop_border', str(crop_border),
        '--csv_path', str(csv_path)]

    if strict_size:
        cmd.append('--strict_size')
    return cmd


def run_evaluation(
    model_name: str,
    sr_dir: Path,
    gt_dir: Path,
    crop_border: int,
    csv_path: Path,
    strict_size: bool = False,
) -> pd.DataFrame:
    """Run the repository evaluation script and return its CSV as a DataFrame."""
    cmd = build_eval_command(
        script_path=EVAL_SCRIPT,
        folder_sr=sr_dir,
        folder_gt=gt_dir,
        crop_border=crop_border,
        csv_path=csv_path,
        strict_size=strict_size)

    print(f'\n[{model_name}]')
    print('Running command:')
    print(' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

    df = pd.read_csv(csv_path)
    df.insert(0, 'model', model_name)
    return df


def build_visual_command(
    script_path: Path,
    folder_sr: Path,
    folder_lq: Path,
    folder_gt: Path,
    seed: int | None = None,
    save_path: Path | None = None,
) -> list[str]:
    """Build the qualitative visualization command."""
    cmd = [
        'python',
        str(script_path),
        '--folder_sr', str(folder_sr),
        '--folder_lq', str(folder_lq),
        '--folder_gt', str(folder_gt)]

    if seed is not None:
        cmd += ['--seed', str(seed)]
    if save_path is not None:
        cmd += ['--save_path', str(save_path)]
    return cmd


def run_visualization(
    model_name: str,
    folder_sr: Path,
    folder_lq: Path,
    folder_gt: Path,
    seed: int | None = None,
    save_path: Path | None = None,
) -> None:
    """Run the random LR / SR / GT triplet visualization script."""
    cmd = build_visual_command(
        script_path=VIS_SCRIPT,
        folder_sr=folder_sr,
        folder_lq=folder_lq,
        folder_gt=folder_gt,
        seed=seed,
        save_path=save_path)

    print(f'\n[Visualization: {model_name}]')
    print('Running command:')
    print(' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)



## Run evaluation

Evaluate every result directory in `MODEL_DIRS`, store one CSV per model,
and concatenate all records into one long-format DataFrame.



In [ ]:
all_results = pd.concat(
    [
        run_evaluation(
            model_name=model_name,
            sr_dir=sr_dir,
            gt_dir=GT_DIR,
            crop_border=CROP_BORDER,
            csv_path=RESULTS_DIR / f'{slugify(model_name)}_eval_pdok.csv',
            strict_size=STRICT_SIZE)
        for model_name, sr_dir in MODEL_DIRS.items()],
    ignore_index=True)

all_results.to_csv(RESULTS_CSV, index=False)
print(f'\nSaved combined evaluation records to: {RESULTS_CSV}')
all_results.head()



## Summary table

Aggregate the per-image results into a compact comparison table. Infinite
PSNR values are excluded from the PSNR aggregates.



In [ ]:
finite = all_results.replace([np.inf, -np.inf], np.nan)

summary = (
    finite
    .groupby('model')
    .agg(
        psnr_mean=('psnr_rgb', 'mean'),
        psnr_min=('psnr_rgb', 'min'),
        psnr_max=('psnr_rgb', 'max'),
        ssim_mean=('ssim_rgb', 'mean'),
        ssim_min=('ssim_rgb', 'min'),
        ssim_max=('ssim_rgb', 'max'),
        n=('image', 'count'))
    .round(4)
    .sort_values(['psnr_mean', 'ssim_mean'], ascending=False))

summary



## Optional detailed inspection

Show the lowest-PSNR samples across all evaluated models. This is often a
useful starting point for manual error analysis.



In [ ]:
all_results.sort_values('psnr_rgb').head(10)



## Optional qualitative check

Metrics are useful, but SR quality should also be inspected visually. The
cell below creates one random LR / SR / GT comparison figure for a selected
model using the existing repository utility.



In [ ]:
run_visualization(
    model_name=VIS_MODEL,
    folder_sr=MODEL_DIRS[VIS_MODEL],
    folder_lq=LR_DIR,
    folder_gt=GT_DIR,
    seed=VIS_SEED,
    save_path=VIS_SAVE_PATH)


## Notes on outputs

- Per-model CSV files are saved in `RESULTS_DIR`.
- The combined CSV is saved as `swinir_evaluation_results.csv`.
- The optional qualitative figure is saved to `VIS_SAVE_PATH`.
- The evaluation utility matches SR and GT filenames by stem and strips
  common suffixes such as `_SwinIR` and `_x4` when needed.

This keeps the notebook aligned with the scripts already used in the
original SwinIR workflow.



---
